# Python — Generators & List Comprehensions
**Day 1 — Python Module**

---

<div style="padding:15px;border-left:8px solid #667eea;background:#f0f0ff;border-radius:4px;">
<strong>Core Insight:</strong> Generators are <em>lazy</em> — they produce one value at a time instead
of building the whole result in memory. For data engineering with millions of rows, this is
the difference between crashing and streaming. A generator expression on 10 million items
uses ~200 bytes. A list comprehension uses ~80 MB.
</div>

### Why It Matters
In data pipelines, you rarely need all records in memory at once. You need to process them one at a time (or in chunks). Generators are Python's native streaming mechanism.

## List Comprehensions — Fast and Readable

```python
# Standard pattern: [expression for item in iterable if condition]
squares = [x**2 for x in range(10) if x % 2 == 0]
# → [0, 4, 16, 36, 64]

# Nested: flatten a 2D list
matrix = [[1, 2], [3, 4], [5, 6]]
flat = [num for row in matrix for num in row]
# → [1, 2, 3, 4, 5, 6]

# Dict comprehension
word_lengths = {word: len(word) for word in ["data", "engineer", "SQL"]}
# → {'data': 4, 'engineer': 8, 'SQL': 3}

# Set comprehension (deduplication)
unique_lengths = {len(word) for word in ["data", "engineer", "SQL", "cat"]}
# → {3, 4, 8}
```

**Rule:** Use list comprehension when you need the full list in memory at once (small data, or data you'll iterate multiple times). Use a generator when processing once in sequence.

In [ ]:
# Memory comparison — list vs generator

import sys

# List comprehension: builds the entire list in memory
squares_list = [x**2 for x in range(10_000_000)]
print(f"List size:      {sys.getsizeof(squares_list):>15,} bytes")  # ~80 MB

# Generator expression: lazy, holds state only
squares_gen = (x**2 for x in range(10_000_000))
print(f"Generator size: {sys.getsizeof(squares_gen):>15,} bytes")   # ~208 bytes

# Both iterate identically — but generator can only go forward, once
total_gen  = sum(squares_gen)            # consumes the generator
total_list = sum(squares_list)           # list is reusable
print(f"Equal totals: {total_gen == total_list}")  # True

## Generator Functions — `yield`

A generator function uses `yield` instead of `return`. When called, it returns a generator object — an iterator that runs the function body up to the next `yield`, pauses, and resumes on the next call.

**Execution model:**
1. Call the function → get a generator object (nothing runs yet)
2. `next(gen)` → runs until `yield`, suspends, returns the yielded value
3. `next(gen)` → resumes from after `yield`, runs until next `yield`
4. When function returns (or falls off the end) → raises `StopIteration`

In [ ]:
# Generator function — process a large file in chunks
# Never loads more than chunk_size rows into memory

def read_csv_chunks(filepath: str, chunk_size: int = 1000):
    """Yield lists of lines in chunks. Constant memory regardless of file size."""
    with open(filepath, encoding='utf-8') as f:
        chunk = []
        for line in f:
            chunk.append(line.rstrip())
            if len(chunk) == chunk_size:
                yield chunk        # pause, hand chunk to caller
                chunk = []         # reset — old chunk is GC'd
        if chunk:
            yield chunk            # yield the final partial chunk

# Usage: never more than 1000 rows in memory
# for chunk in read_csv_chunks("telemetry_10gb.csv", chunk_size=1000):
#     process_chunk(chunk)    # process 1000 rows, then release them

# Demonstrate with a simple counter generator
def counter_gen(n):
    """Yields 0, 1, 2, ..., n-1"""
    i = 0
    while i < n:
        yield i
        i += 1

gen = counter_gen(5)
print(next(gen))   # 0  — runs until yield, pauses
print(next(gen))   # 1
print(list(gen))   # [2, 3, 4] — consume the rest
# next(gen)        # → StopIteration (generator exhausted)

In [ ]:
# Real pipeline: generator chaining
# Process a stream of server metrics without loading everything

def parse_line(line: str) -> dict:
    """Parse a CSV line into a metric dict."""
    parts = line.split(',')
    return {'server_id': parts[0], 'metric': parts[1], 'value': float(parts[2])}

def filter_high_cpu(records, threshold=80):
    """Generator: yield only records above threshold."""
    for rec in records:
        if rec['metric'] == 'cpu_pct' and rec['value'] > threshold:
            yield rec

def enrich_with_tier(records, server_tiers: dict):
    """Generator: add tier information to each record."""
    for rec in records:
        rec['tier'] = server_tiers.get(rec['server_id'], 'unknown')
        yield rec

# Chain generators — each step lazy, zero intermediate lists
# lines       = open("metrics.csv")              # lazy file read
# parsed      = (parse_line(l) for l in lines)   # lazy parse
# high_cpu    = filter_high_cpu(parsed, 80)       # lazy filter
# enriched    = enrich_with_tier(high_cpu, tiers) # lazy enrich
# results     = list(enriched)                    # materialize ONLY here

# This processes a 10GB file using only ~O(1) memory at each step.
print("Generator pipeline demo (no actual file needed)")
print("Each step is lazy — data flows one record at a time")

## itertools — Production-Grade Iteration

`itertools` provides lazy combinators for generators. Key tools for data engineering:

In [ ]:
import itertools

# chain: combine multiple iterables lazily (no intermediate list)
servers   = ["srv-01", "srv-02", "srv-03"]
databases = ["db-01", "db-02"]
all_resources = list(itertools.chain(servers, databases))
print("chain:", all_resources)   # ['srv-01', 'srv-02', 'srv-03', 'db-01', 'db-02']

# islice: take the first N items from a generator (safe — doesn't exhaust it)
def infinite_counter(start=0):
    while True:
        yield start
        start += 1

first_five = list(itertools.islice(infinite_counter(100), 5))
print("islice:", first_five)    # [100, 101, 102, 103, 104]

# groupby: group consecutive items (sort first! groupby is NOT a SQL GROUP BY)
from itertools import groupby

data = sorted([
    ("production", "srv-01"), ("production", "srv-02"),
    ("staging",    "srv-03"), ("dev",        "srv-04"),
], key=lambda x: x[0])

for env, items in groupby(data, key=lambda x: x[0]):
    servers_in_env = [s for _, s in items]
    print(f"  {env}: {servers_in_env}")

# batched (Python 3.12+): split an iterable into batches of N
# For older Python, use islice in a loop:
def batched(iterable, n):
    it = iter(iterable)
    while batch := list(itertools.islice(it, n)):
        yield batch

for batch in batched(range(10), 3):
    print("batch:", batch)  # [0,1,2], [3,4,5], [6,7,8], [9]

## Interview Q&A

**Q: What's the difference between a list comprehension and a generator expression?**
A: A list comprehension `[x for x in ...]` builds the full list in memory immediately — eager evaluation. A generator expression `(x for x in ...)` is lazy — yields one value at a time. Use generators for large data where you only need one pass.

**Q: When does `yield` pause execution?**
A: Exactly at the `yield` statement. The function's local state (all variables, the position in the code) is saved. On the next `next()` call, execution resumes from the line after `yield`.

**Q: How would you process a 10GB CSV without loading it into memory?**
A: Use a generator that reads and yields rows (or chunks) one at a time. `pandas.read_csv(filepath, chunksize=1000)` is built on this exact pattern — it returns an iterator of DataFrames, each with 1000 rows.

**Q: What is a generator's key limitation vs a list?**
A: Generators are single-pass and forward-only. Once an element is yielded and consumed, you can't go back. If you need to iterate the data multiple times (e.g., compute mean, then compute variance), materialize to a list first — or use two separate generators.

**Q: What is `itertools.groupby` and what's the gotcha?**
A: `groupby` groups consecutive equal elements. The gotcha: it only groups consecutive items, not all items with the same key (unlike SQL GROUP BY). You **must sort by the grouping key first** or you'll get multiple groups for the same key.

## The Citi Angle

**At Citi, generators were the right tool for telemetry pipelines.** APM data from 6,000 servers generated millions of metric records per collection cycle. Loading everything into memory was never an option.

**The chunk-processing pattern:**
```python
def process_telemetry_file(path: str, batch_size: int = 5000):
    """Process telemetry CSV in memory-safe batches."""
    for chunk in read_csv_chunks(path, batch_size):
        # Each chunk is a list of 5000 lines
        records = [parse_metric(line) for line in chunk]
        records = [r for r in records if r['cpu_pct'] > 70]  # filter
        if records:
            insert_to_db(records)   # batch insert, not row-by-row
        # chunk goes out of scope, memory freed immediately
```

**Interview line:** *"For capacity data processing at Citi — 6,000 servers, millions of rows per cycle — Python generators were essential. The chunk pattern kept memory constant while processing arbitrarily large files. Same mental model as Spark's lazy execution: don't materialize until you have to."*